# Session 0 — Prepare Google Drive before the tutorial

Run this notebook **once before the workshop**. It stores the ~350 MB Tonsil RNA/ADT
pair and a Python wheel cache in `MyDrive/ECCB2026`. Later session notebooks use a new
Colab runtime when necessary, but reuse these persistent files instead of downloading
them again from the internet.

This notebook does not train DGAT and does not install the full tutorial environment.

Before running it, select **Runtime → Change runtime type → 2026.04** (Python 3.12).
Use the same runtime version for Sessions 1–3 so the cached wheels remain compatible.


## 1. Mount Drive and fetch the small tutorial repository


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Open this notebook in Google Colab to prepare Google Drive.")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

repo_dir = Path("/content/ECCB-2026-Tutorial")
tutorial_root = repo_dir / "hands-on_tutorial"
if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)],
        check=True,
    )
os.chdir(tutorial_root)
print("Tutorial repository:", tutorial_root)


## 2. Download the Tonsil assets directly into Drive


In [ ]:
if importlib.util.find_spec("gdown") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown>=5"], check=True)

drive_root = Path("/content/drive/MyDrive/ECCB2026")
asset_root = drive_root / "assets" / "DGAT_assets"
asset_root.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment["DGAT_ASSET_DIR"] = str(asset_root)
command = ["bash", "scripts/download_dgat_assets.sh", "--data-only", "--dataset", "Tonsil"]
subprocess.run(command, cwd=tutorial_root, env=environment, check=True)
subprocess.run(command + ["--check-only"], cwd=tutorial_root, env=environment, check=True)


## 3. Verify the downloads and record checksums


In [ ]:
import hashlib
import json

data_dir = asset_root / "data"
manifest = {"python": sys.version.split()[0], "files": {}}
for filename in ("Tonsil_RNA.h5ad", "Tonsil_ADT.h5ad"):
    path = data_dir / filename
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty asset: {path}")
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    manifest["files"][filename] = {
        "bytes": path.stat().st_size,
        "sha256": digest.hexdigest(),
    }

manifest_path = drive_root / "asset_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(json.dumps(manifest, indent=2))


## 4. Cache the small Session 1 Python environment in Drive


In [ ]:
wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"
wheelhouse.mkdir(parents=True, exist_ok=True)
requirements = tutorial_root / "requirements-colab.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "download", "-q", "--only-binary=:all:",
     "--dest", str(wheelhouse), "-r", str(requirements)],
    check=True,
)
print(f"Cached {len(list(wheelhouse.glob('*.whl')))} wheels in {wheelhouse}")


## 5. Create persistent state folders


In [ ]:
for relative in ("state/data/processed", "state/results/figures", "state/checkpoints"):
    (drive_root / relative).mkdir(parents=True, exist_ok=True)

print("Drive preparation complete:", drive_root)
print("You may close this runtime. Open the Session 1 notebook when the workshop begins.")
